# Circular QR Code — Apple Minimalist Aesthetic
Orbiting dots seamlessly encircle a central logo with refined, Apple-inspired precision.

In [ ]:
!pip install qrcode[pil] pillow numpy -q

In [ ]:
import qrcode
from PIL import Image, ImageDraw, ImageFilter
import math
import numpy as np


def _hex_to_rgb(color):
    """Convert hex string or RGB tuple to (R, G, B)."""
    if isinstance(color, tuple):
        return color[:3]
    color = color.strip()
    if color.startswith('#'):
        color = color[1:]
    return tuple(int(color[i:i+2], 16) for i in (0, 2, 4))


def _draw_orbit_dot(draw, x, y, size, color_rgba, shape='circle'):
    """Draw a single orbit dot in circle, square, or diamond shape."""
    r = size // 2
    if shape == 'circle':
        draw.ellipse([x - r, y - r, x + r, y + r], fill=color_rgba)
    elif shape == 'square':
        draw.rectangle([x - r, y - r, x + r, y + r], fill=color_rgba)
    elif shape == 'diamond':
        pts = [(x, y - r), (x + r, y), (x, y + r), (x - r, y)]
        draw.polygon(pts, fill=color_rgba)


def create_circular_qr(
    link: str,
    output_path: str = 'qr_apple.png',
    # QR appearance
    fill_color: str = '#1d1d1f',
    back_color: str = '#f5f5f7',
    box_size: int = 10,
    border: int = 4,
    # Canvas layout
    circle_padding: int = 30,
    background_color: tuple = None,
    # Outer ring
    add_outer_ring: bool = True,
    outer_ring_color: str = '#d2d2d7',
    outer_ring_width: int = 6,
    # Center logo
    logo_path: str = None,
    logo_size_ratio: float = 0.22,
    # Orbit dots
    orbit_dot_count: int = 16,
    orbit_dot_size: int = 10,
    orbit_gap: int = 18,
    orbit_dot_color: str = None,
    orbit_dot_shape: str = 'circle',   # 'circle', 'square', 'diamond'
    orbit_fade: bool = True,           # Fade bottom dots for depth
    orbit_rotation_offset: float = 0.0,  # Degrees — rotate the dot ring
):
    """
    Generate an Apple-aesthetic circular QR code with a seamless
    ring of orbiting dots around the perimeter.

    Args:
        link:                URL or text to encode.
        output_path:         Output PNG file path.
        fill_color:          QR module color (hex or name).
        back_color:          QR background color.
        box_size:            Pixels per QR module.
        border:              Quiet-zone modules.
        circle_padding:      Extra pixels beyond the QR square.
        background_color:    RGBA tuple for the circular bg. Defaults to back_color.
        add_outer_ring:      Draw a thin decorative ring at the edge.
        outer_ring_color:    Color of the outer ring.
        outer_ring_width:    Pixel width of the outer ring.
        logo_path:           Path to center logo PNG (transparency supported).
        logo_size_ratio:     Logo diameter as fraction of canvas.
        orbit_dot_count:     Number of dots orbiting the QR.
        orbit_dot_size:      Diameter of each orbit dot in pixels.
        orbit_gap:           Gap in pixels between QR edge and dot centers.
        orbit_dot_color:     Dot color (defaults to fill_color).
        orbit_dot_shape:     'circle', 'square', or 'diamond'.
        orbit_fade:          Fade lower dots for a shadow/depth effect.
        orbit_rotation_offset: Rotate the entire dot ring by N degrees.

    Returns:
        PIL.Image RGBA result.
    """

    if background_color is None:
        bg_rgb = _hex_to_rgb(back_color)
        background_color = bg_rgb + (255,)

    if orbit_dot_color is None:
        orbit_dot_color = fill_color

    orbit_rgb = _hex_to_rgb(orbit_dot_color)
    ring_rgb  = _hex_to_rgb(outer_ring_color)

    # ── 1. Build QR ─────────────────────────────────────────────────────────
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=box_size,
        border=border,
    )
    qr.add_data(link)
    qr.make(fit=True)
    qr_img  = qr.make_image(fill_color=fill_color, back_color=back_color).convert('RGBA')
    qr_w, qr_h = qr_img.size
    size = max(qr_w, qr_h)

    # ── 2. Canvas ────────────────────────────────────────────────────────────
    canvas_size = size + circle_padding * 2
    canvas = Image.new('RGBA', (canvas_size, canvas_size), (0, 0, 0, 0))
    bg_layer = Image.new('RGBA', (canvas_size, canvas_size), (0, 0, 0, 0))
    bg_draw  = ImageDraw.Draw(bg_layer)
    radius   = canvas_size // 2
    cx = cy  = canvas_size // 2

    # Solid background circle
    bg_draw.ellipse([cx - radius, cy - radius, cx + radius - 1, cy + radius - 1],
                    fill=background_color)

    # Outer ring
    if add_outer_ring:
        rw = outer_ring_width
        for t in range(rw):
            r_t = radius - t - 1
            bg_draw.ellipse(
                [cx - r_t, cy - r_t, cx + r_t, cy + r_t],
                outline=ring_rgb + (255,),
                width=1
            )

    canvas = Image.alpha_composite(canvas, bg_layer)

    # ── 3. Make QR transparent background then paste ─────────────────────────
    back_rgb_val = _hex_to_rgb(back_color)
    arr = np.array(qr_img)
    tol = 30
    is_back = (
        (np.abs(arr[:,:,0].astype(int) - back_rgb_val[0]) < tol) &
        (np.abs(arr[:,:,1].astype(int) - back_rgb_val[1]) < tol) &
        (np.abs(arr[:,:,2].astype(int) - back_rgb_val[2]) < tol)
    )
    arr[:,:,3] = np.where(is_back, 0, 255)
    qr_rgba = Image.fromarray(arr, 'RGBA')

    ox = (canvas_size - qr_w) // 2
    oy = (canvas_size - qr_h) // 2
    canvas.paste(qr_rgba, (ox, oy), mask=qr_rgba)

    # ── 4. Circular crop ─────────────────────────────────────────────────────
    circle_mask = Image.new('L', (canvas_size, canvas_size), 0)
    ImageDraw.Draw(circle_mask).ellipse([0, 0, canvas_size - 1, canvas_size - 1], fill=255)
    result = Image.new('RGBA', (canvas_size, canvas_size), (0, 0, 0, 0))
    result.paste(canvas, mask=circle_mask)

    # ── 5. Orbiting dots ─────────────────────────────────────────────────────
    dot_draw   = ImageDraw.Draw(result)
    qr_radius  = size / 2
    dot_orbit  = qr_radius + orbit_gap
    rot_offset = math.radians(orbit_rotation_offset - 90)  # start from top

    for i in range(orbit_dot_count):
        angle = rot_offset + (i / orbit_dot_count) * math.tau
        dx = int(cx + dot_orbit * math.cos(angle))
        dy = int(cy + dot_orbit * math.sin(angle))

        alpha = 255
        if orbit_fade:
            # Dots near bottom (angle ≈ π/2) are dimmer
            norm  = (math.sin(angle + math.pi / 2) + 1) / 2  # 0..1, top=1
            alpha = int(64 + 191 * norm)                      # 64..255

        color_rgba = orbit_rgb + (alpha,)
        _draw_orbit_dot(dot_draw, dx, dy, orbit_dot_size, color_rgba, orbit_dot_shape)

    # ── 6. Center logo / dot ─────────────────────────────────────────────────
    if logo_path:
        try:
            logo = Image.open(logo_path).convert('RGBA')
            logo_d   = int(canvas_size * logo_size_ratio)
            logo_r   = logo_d // 2
            logo_bg  = Image.new('RGBA', (logo_d, logo_d), (0, 0, 0, 0))
            lbg_draw = ImageDraw.Draw(logo_bg)

            # Subtle shadow ring
            so = max(2, logo_d // 40)
            lbg_draw.ellipse([so, so, logo_d - 1, logo_d - 1], fill=(0, 0, 0, 60))
            lbg_draw.ellipse([0, 0, logo_d - so - 1, logo_d - so - 1], fill=(255, 255, 255, 255))

            pad       = max(4, logo_d // 8)
            inner     = logo_d - pad * 2
            logo_rs   = logo.resize((inner, inner), Image.LANCZOS)
            clip      = Image.new('L', (inner, inner), 0)
            ImageDraw.Draw(clip).ellipse([0, 0, inner - 1, inner - 1], fill=255)
            logo_rs.putalpha(clip)
            logo_bg.paste(logo_rs, (pad, pad), mask=logo_rs)

            result.paste(logo_bg, (cx - logo_r, cy - logo_r), mask=logo_bg)
            print(f'Logo added from: {logo_path}')
        except FileNotFoundError:
            print(f'Logo not found at "{logo_path}" — falling back to center dot.')
            logo_path = None
        except Exception as e:
            print(f'Could not load logo: {e} — falling back to center dot.')
            logo_path = None

    if not logo_path:
        # Apple-style minimal center: small circle with a dot
        center_d = max(20, int(canvas_size * 0.075))
        center_r = center_d // 2
        cd = ImageDraw.Draw(result)
        bg_rgb = _hex_to_rgb(back_color)
        cd.ellipse([cx - center_r, cy - center_r, cx + center_r, cy + center_r],
                   fill=bg_rgb + (255,), outline=ring_rgb + (255,), width=2)
        dot_r = max(4, center_d // 6)
        fill_rgb = _hex_to_rgb(fill_color)
        cd.ellipse([cx - dot_r, cy - dot_r, cx + dot_r, cy + dot_r],
                   fill=fill_rgb + (255,))

    # ── 7. Save ───────────────────────────────────────────────────────────────
    result.save(output_path, format='PNG')
    print(f'Saved: {output_path}')
    return result


In [ ]:
# ── Apple Minimal (default) ──────────────────────────────────────────────────
from IPython.display import display

LINK = 'https://forms.gle/UUDbRHLBSQp5McmZA'

img = create_circular_qr(
    link=LINK,
    output_path='apple_minimal.png',
    logo_path='logo.png',           # Optional — comment out if no logo
    fill_color='#1d1d1f',
    back_color='#f5f5f7',
    outer_ring_color='#d2d2d7',
    orbit_dot_count=16,
    orbit_dot_size=10,
    orbit_gap=18,
    orbit_dot_shape='circle',
    orbit_fade=True,
)
display(img)

In [ ]:
# ── Dark / Midnight variant ───────────────────────────────────────────────────
img_dark = create_circular_qr(
    link=LINK,
    output_path='apple_dark.png',
    fill_color='#f5f5f7',
    back_color='#1d1d1f',
    outer_ring_color='#3a3a3c',
    orbit_dot_color='#f5f5f7',
    orbit_dot_count=20,
    orbit_dot_size=8,
    orbit_gap=16,
    orbit_dot_shape='circle',
    orbit_fade=True,
)
display(img_dark)

In [ ]:
# ── Diamond dots — editorial accent ──────────────────────────────────────────
img_diamond = create_circular_qr(
    link=LINK,
    output_path='apple_diamond.png',
    fill_color='#1d1d1f',
    back_color='#ffffff',
    outer_ring_color='#000000',
    outer_ring_width=3,
    orbit_dot_count=24,
    orbit_dot_size=7,
    orbit_gap=14,
    orbit_dot_shape='diamond',
    orbit_fade=False,
    orbit_rotation_offset=7.5,  # offset so dots align between corner finders
)
display(img_diamond)